In [1]:
from st_router.paths import get_rootdir
from st_router.scoring import label_rows, score, run_mcnemar
from st_router.prompt import make_prompt, run_llm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from dotenv import load_dotenv
import groq
from groq import Groq
from time import sleep
import re
import logging
logging.basicConfig(level=logging.INFO)


ROOT_DIR = get_rootdir()
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROC = ROOT_DIR / "data" / "processed"
DATA_INTERIM = ROOT_DIR / "data" / "interim"

load_dotenv()

True

# Part 2: Empty Prompt
Next-cmd is ran while cli is empty, only past 10 commands exist

In [2]:
bash = pd.read_parquet(DATA_PROC / "bash_windows.parquet")

df = bash.copy()
df = df[df["window_len"] != 0]

# randomly select one row per user - 267 users
df = df.groupby("unique_key").sample(random_state=42)
df.head()

,unique_key,timestamp,hostname,cmd,wd,cmd_type,training,gap_s,session_exceed,session,...,prev_2,prev_3,prev_4,prev_5,prev_6,prev_7,prev_8,prev_9,prev_10,window_len
2,House of cards_0_101,2019-09-26 09:24:40.283127+00:00,attacker,nmap -sV 172.18.1.5,/root,bash-command,House of cards,46.831211,False,0,...,nmap -sV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
21,House of cards_0_112,2019-09-19 11:02:05.597042+00:00,attacker,ssh 172.18.1.5 -l alice,/root,bash-command,House of cards,7.933765,False,0,...,msf5 --help,nmap -Pn --script vuln 172.18.1.5,nmap -sV 172.18.1.5,nmap --help,nmap --script vuln 172.18.1.5,nmap -v -script vulscan 172.18.1.5,nmap -v -script vuln -p22 172.18.1.5,nmap -sV -p22 172.18.1.5,nmap -sV 172.18.1.5 -A -v,10
39,House of cards_0_113,2019-09-19 10:34:43.005975+00:00,attacker,nmap -p 1-56000 172.18.1.5,/root,bash-command,House of cards,54.440345,False,0,...,nmap -p 1-1000 172.18.1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
55,House of cards_0_115,2019-09-19 10:38:09.352460+00:00,attacker,ssh --p,/root,bash-command,House of cards,36.140902,False,0,...,ssh,libssh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
76,House of cards_0_116,2019-09-19 10:53:16.176064+00:00,attacker,run,/root,bash-command,House of cards,1.288221,False,0,...,set RPORT ~,set RHOST 172.18.1.5,sudo service metasploit start,sudo service postgresql start,msfconsole,mfsconsole,mfconsole,db_nmap -v -sV 172.18.1.5,ping 172.18.1.5,10


In [ ]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
prompt = make_prompt()

out20 = DATA_INTERIM / "empty_prompt_gpt20b.jsonl"
if not out20.exists():
    run_llm(df, prompt, out20)

out120 = DATA_INTERIM / "empty_prompt_gptoss120b.jsonl"
if not out120.exists():
    run_llm(df, prompt, out120, model="openai/gpt-oss-120b")

out_gem = DATA_INTERIM / "empty_prompt_gemini35flash_lite_low.jsonl"
if not out_gem.exists():
    run_llm(df, prompt, out_gem, model="gemini-3.5-flash-lite", provider="gemini", reasoning_effort=None, sleep_s=5)

# Exploratory

In [8]:
df = pd.read_json(DATA_INTERIM / "base_rate_gptoss20b.jsonl", lines=True)
df["prediction"] = df["prediction"].str.strip()
df["actual"] = df["actual"].str.strip()
accuracy = (df["actual"] == df["prediction"]).sum()
print(f"exact prediction accuracy {accuracy/df.shape[0]*100}%")

exact prediction accuracy 6.741573033707865%


Exact matching is very harsh with only 9% accuracy rate on the test set. We need to investigate what it gets wrong

In [9]:
print(df["prediction"].isna().sum())
print(df["actual"].isna().sum())
# one space
print((df["prediction"] == " ").sum())
print((df["actual"] == " ").sum())
# empty string      
print((df["prediction"] == "").sum())
print((df["actual"] == "").sum())

display(df["prediction"].value_counts())
display(df["actual"].value_counts())

0
0
0
0
0
0


prediction
ls -la                                      22
ls                                          14
nmap -sV 172.18.1.5                         11
ls -l                                        9
nmap -A 172.18.1.5                           8
                                            ..
python3 sshjohn.py                           1
nmap -v 192.168.128.0/24                     1
crc32 file.txt                               1
ls -l $(find / -name metasploit -type d)     1
ssh-keygen -t rsa -b 4096 -f keys/id_rsa     1
Name: count, Length: 176, dtype: int64

actual
ls                                                                                     28
msfconsole                                                                              7
nmap -sV 172.18.1.5                                                                     6
pwd                                                                                     6
cd ..                                                                                   5
                                                                                       ..
cp $(locate sshjohn.py) .                                                               1
nc -l -p 51071 > id_rsa                                                                 1
nmap -v 10.1.26.9                                                                       1
cat /usr/share/metasploit-framework/lib/msf/core/modules/external/python/metasploit     1
cd ~/exploit/unix/webapp                                                                1
Nam

Exact matches not a good metric. a lot of the commands entered in actual aren't predictable e.g passwords, typos. We rely on the work of Davison & Hirsh (1998), "Predicting Sequences of User Actions", Korvemaker & Greiner (2000), "Predicting UNIX Command Lines: Adjusting to User Patterns", Lin et al. (2018), NL2Bash.

We need to count how many of the misses are: 1) same command with different flags/arguments 2) different command

In [10]:
cmd_pred = df["prediction"].str.replace(r"^sudo\s+", "", regex=True).str.split().str[0]
cmd_actual = df["actual"].str.replace(r"^sudo\s+", "", regex=True).str.split().str[0]

print(f"command matches: {(cmd_pred == cmd_actual).mean() * 100}%")

command matches: 31.835205992509362%


In [11]:
match = cmd_pred == cmd_actual
display(cmd_actual[match].value_counts())

actual
nmap          36
ls            18
ssh            7
fcrackzip      4
python         3
sqlmap         3
john           2
cd             1
find           1
scp            1
unzip          1
pwd            1
ifconfig       1
ssh-add        1
cat            1
nano           1
msfconsole     1
dpkg           1
nc             1
Name: count, dtype: int64

We decide to score:
+ Exact: compare full string
+ Command-name: strip whitespace, strip sudo, compare first token (command)

Three target cateogies: exact, edited, different

## TODO
As we scale a lookup table will classify if a command is valid and check if it is a typo (within 1 Damerau–Levenshtein distance), or invalid command e.g. password. Typos in the actual values will be then scored against the actual predicted commands by using the lookup table for the precise syntax.

This will be done by eye in our small test set

In [12]:
with pd.option_context("display.max_rows", None):
    display(cmd_actual[~match].value_counts())

actual
cd                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    28
ls                                                                                                                                                                                                                                      

# Scoring

2 clean typos (mfsconsole, ipconfig) of 10 problem rows

Main limitation: smart-terminal has more context than the model. We only give it the working direcotyr while smart terminal has:

+ cwd_contents: the files and folders in the current directory. This is the big one.
+ shell_tools (tool versions), plus shell and os.
+ Memory of past suggestions in that project.
+ Tools, in particular ReadLastError, which reads the last error message.

# Evaluation

In [ ]:
from itertools import combinations
import numpy as np

FILES = {
    "gptoss20b":  DATA_INTERIM / "empty_prompt_gptoss20b.jsonl",
    "gptoss120b": DATA_INTERIM / "empty_prompt_gptoss120b.jsonl",
    "gemini35fl": DATA_INTERIM / "empty_prompt_gemini35flash_lite_low.jsonl",  
}

def load_scored(path):
    df = pd.read_json(path, lines=True)
    df = label_rows(df)
    df = df[df["label"] != "non_command"].copy()
    assert df["idx"].is_unique, f"duplicate idx in {path.name}"
    df = df.set_index("idx").sort_index()
    exact, cmd = score(df["prediction"], df["target"])
    df["exact"] = np.asarray(exact, dtype=bool)
    df["cmd"] = np.asarray(cmd, dtype=bool)
    return df[["target", "exact", "cmd"]]

results = {name: load_scored(path) for name, path in FILES.items()}

# Same rows and same targets in every file
ref = next(iter(results.values()))
for name, r in results.items():
    assert r.index.equals(ref.index), f"{name}: different rows"
    assert r["target"].equals(ref["target"]), f"{name}: different targets"

# Baseline: repeat last command, on the same rows
prev_1 = bash.loc[ref.index, "prev_1"].values
b_exact, b_cmd = score(pd.Series(prev_1, index=ref.index), ref["target"])
results["baseline"] = pd.DataFrame(
    {"target": ref["target"],
     "exact": np.asarray(b_exact, dtype=bool),
     "cmd": np.asarray(b_cmd, dtype=bool)},
    index=ref.index,
)

print(f"n = {len(ref)}")
for name, r in results.items():
    print(f"{name:11s} exact {r['exact'].mean():6.1%}   command {r['cmd'].mean():6.1%}")

for a, b in combinations(results, 2):
    print(f"\n{a} vs {b}")
    run_mcnemar(results[a]["exact"], results[b]["exact"], "exact")
    run_mcnemar(results[a]["cmd"], results[b]["cmd"], "command")

n = 257
gptoss20b   exact   7.0%   command  33.1%
gptoss120b  exact  10.1%   command  33.1%
gemini35fl  exact  12.1%   command  35.0%
baseline    exact   3.1%   command  31.5%

gptoss20b vs gptoss120b
exact: both right=12, only model=6, only baseline=14, both wrong=225, p=0.115
command: both right=69, only model=16, only baseline=16, both wrong=156, p=1.000

gptoss20b vs gemini35fl
exact: both right=16, only model=2, only baseline=15, both wrong=224, p=0.002
command: both right=69, only model=16, only baseline=21, both wrong=151, p=0.511

gptoss20b vs baseline
exact: both right=1, only model=17, only baseline=7, both wrong=232, p=0.064
command: both right=51, only model=34, only baseline=30, both wrong=142, p=0.708

gptoss120b vs gemini35fl
exact: both right=17, only model=9, only baseline=14, both wrong=217, p=0.405
command: both right=66, only model=19, only baseline=24, both wrong=148, p=0.542

gptoss120b vs baseline
exact: both right=0, only model=26, only baseline=8, both wrong=22